# Warning :
# Do "File -> Save a copy in Drive" before you start modifying the notebook, otherwise your modifications will not be saved.


In [ ]:
import argparse
import os
import time

import PIL
from PIL import Image

import numpy as np
import torchvision
import pickle

import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.utils.data
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torch.autograd import Variable

from sklearn.svm import LinearSVC

# Partie 1 : Architecture VGG16

In [ ]:
#!wget https://github.com/cdancette/deep-learning-polytech-tp6-7/raw/master/tp8/imagenet_classes.pkl
!wget https://github.com/rdfia/rdfia.github.io/raw/master/data/3-a/imagenet_classes.pkl

# Bonus : Classifiez des exemples avec vgg16 et commentez le résultat dans votre rapport.
!wget --content-disposition https://unsplash.com/photos/gKXKBY-C-Dk/download?force=true -O cat.jpg
!wget --content-disposition https://unsplash.com/photos/qO-PIF84Vxg/download?force=true -O dog.jpg
!wget --content-disposition https://unsplash.com/photos/1bPXERSwOcg/download?force=true -O horse.jpg
!wget --content-disposition https://unsplash.com/photos/c8XlAc1akIU/download?force=true -O bear.jpg

In [ ]:
cat = Image.open('cat.jpg')
dog = Image.open('dog.jpg')
bear = Image.open('bear.jpg')
horse = Image.open('horse.jpg')

IMG = [cat, dog, bear, horse]

In [ ]:
# Loading ImageNet classes
imagenet_classes = pickle.load(open('imagenet_classes.pkl', 'rb'))

# Créer une figure avec plusieurs sous-figures (par exemple 2 lignes, 2 colonnes)
fig, axs = plt.subplots(3, 4, figsize=(12, 10))

# Aplatir le tableau des axes pour faciliter l'indexation
axs = axs.flatten()

# Boucle sur chaque image
for idx, img in enumerate(IMG):
    axs[idx].imshow(img)
    axs[idx].set_title("Image before normalization")
    axs[idx].axis('off')

    # Normalization
    img = img.resize((224, 224), Image.BILINEAR)
    img = np.array(img, dtype=np.float32) / 255
    img = img.transpose((2, 0, 1))

    # ImageNet mean/std
    mu = torch.Tensor([0.485, 0.456, 0.406])
    sigma = torch.Tensor([0.229, 0.224, 0.225])
    mu = mu[:, None, None]
    sigma = sigma[:, None, None]

    img = torch.tensor(img)
    img = (img - mu) / sigma
    vgg16 = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1)
    vgg16.eval()

    img = np.expand_dims(img, 0)
    x = torch.Tensor(img)
    y = vgg16(x).detach()

    top5_values, top5_indices = torch.topk(y, k=5)
    y = y.numpy()

    y_label = np.argmax(y)
    print(f"The picture represents: {imagenet_classes[y_label]}")

    top5_values = top5_values.squeeze().tolist()
    top5_indices_list = top5_indices.squeeze().tolist()
    top5_classes = [imagenet_classes[idx][:25] for idx in top5_indices_list]

    axs[idx + 4].imshow(img[0].transpose(1, 2, 0))
    axs[idx + 4].set_title(f"Predicted Class: {imagenet_classes[y_label]}")
    axs[idx + 4].axis('off')

    axs[idx + 8].bar(top5_classes, top5_values, color='skyblue')
    axs[idx + 8].set_xticks(top5_classes)
    axs[idx + 8].set_xticklabels(top5_classes, rotation=45, ha='right')
    axs[idx + 8].set_title("Top 5 Predictions")
    axs[idx + 8].set_xlabel("Classes")
    axs[idx + 8].set_ylabel("Values")


plt.tight_layout()

plt.show()


##Activation Maps

In [ ]:
for idx, img in enumerate(IMG):
  vgg16.eval()

  activations = None

  def hook_fn(module, input, output):
      global activations
      activations = output

  vgg16.features[0].register_forward_hook(hook_fn)

  img = img.resize((224, 224), Image.BILINEAR)
  img = np.array(img, dtype=np.float32) / 255
  img = img.transpose((2, 0, 1))

  img = torch.tensor(img)
  img = (img - mu) / sigma
  img = np.expand_dims(img, 0)
  x = torch.Tensor(img)
  y = vgg16(x).detach()

  activation_maps = activations.squeeze(0)

  # Plot several activation maps
  num_maps_to_display = 32
  fig, axes = plt.subplots(4, 8, figsize=(10, 4))

  for idx, ax in enumerate(axes.flat):
      if idx < num_maps_to_display:
          ax.imshow(activation_maps[idx].cpu().detach().numpy(), cmap='viridis')
          ax.axis('off')

  plt.tight_layout()
  plt.show()



# Partie 2: Transfer Learning avec VGG16 sur 15 Scene

In [ ]:
#!wget https://github.com/cdancette/deep-learning-polytech-tp6-7/raw/master/tp8/15ScenesData.zip
!wget https://github.com/rdfia/rdfia.github.io/raw/master/data/3-a/15ScenesData.zip

!unzip 15ScenesData.zip

In [ ]:
ls 15SceneData/test/bedroom/

In [ ]:
class VGG16relu7(nn.Module):
  def __init__(self):
    super(VGG16relu7, self).__init__()
    # Copy the entire convolutional part
    self.features = nn.Sequential( *list(vgg16.features.children()))
    # Keep a piece of the classifier: -2 to stop at relu7
    self.classifier = nn.Sequential(*list(vgg16.classifier.children())[:-2])
  def forward(self, x):
    x = self.features(x)
    x = x.view(x.size(0), -1)
    x = self.classifier(x)
    return x

In [ ]:

PRINT_INTERVAL = 50
CUDA = False

def get_dataset(batch_size, path):

    # This function expands 3 times a gray level image
    # to transform it into an image RGB. Use it with transform.Lambda
    def duplicateChannel(img):
        img = img.convert('L')
        np_img = np.array(img, dtype=np.uint8)
        np_img = np.dstack([np_img, np_img, np_img])
        img = Image.fromarray(np_img, 'RGB')
        return img
    def resizeImage(img):
      return img.resize((224,224), Image.BILINEAR)

    #####################
    ## YOUR CODE HERE  ##
    #####################
    # Add pre-processing
    mu = torch.Tensor([0.485, 0.456, 0.406])
    sigma = torch.Tensor([0.229, 0.224, 0.225])
    train_dataset = datasets.ImageFolder(path+'/train',
        transform=transforms.Compose([ # Pre-processing TODO: duplicateChannel(), resizeImage(), toTensor(), Normalize ()
            duplicateChannel,
            resizeImage,
            transforms.ToTensor(),
            transforms.Normalize(mu,sigma)
        ]))
    val_dataset = datasets.ImageFolder(path+'/test',
        transform=transforms.Compose([ # Pre-processing TODO
            duplicateChannel,
            resizeImage,
            transforms.ToTensor(),
            transforms.Normalize(mu,sigma)
        ]))
    ####################
    ##      END        #
    ####################

    train_loader = torch.utils.data.DataLoader(train_dataset,
                        batch_size=batch_size, shuffle=False, pin_memory=CUDA, num_workers=2)
    val_loader = torch.utils.data.DataLoader(val_dataset,
                        batch_size=batch_size, shuffle=False, pin_memory=CUDA, num_workers=2)

    return train_loader, val_loader

In [ ]:
def extract_features(data, model, batch_size):
    #####################
    ## YOUR CODE HERE  ##
    #####################
    # init features matrices
    X = torch.zeros(len(data), batch_size, 4096)
    y = torch.zeros(len(data), batch_size)
    ####################
    ##      END        #
    ####################

    for i, (input, target) in enumerate(data):
        if i % PRINT_INTERVAL == 0:
            print('Batch {0:03d}/{1:03d}'.format(i, len(data)))
        if CUDA:
            input = input.cuda()
        #####################
        ## YOUR CODE HERE  ##
        #####################
        X[i] = model(input)
        y[i] = target
        ####################
        ##      END        #
        ####################

    return X, y


def main(path="15SceneData", batch_size=8):
    print('Instanciation de VGG16')
    vgg16 = models.vgg16(pretrained=True)

    print('Instanciation de VGG16relu7')
    #####################
    ## YOUR CODE HERE  ##
    #####################
    # Remplacer par le modèle par un réseau tronqué pour faire de la feature extraction
    # On créera une nouvelle classe VGG16relu7 ici
    model = VGG16relu7()
    ####################
    ##      END        #
    ####################
    if CUDA is False:
      model = model.cpu()

    model.eval()
    if CUDA: # si on fait du GPU, passage en CUDA
        cudnn.benchmark = True
        model = model.cuda()

    # On récupère les données
    print('Récupération des données')
    train, test = get_dataset(batch_size, path)

    # Extraction des features
    print('Feature extraction')
    X_train, y_train = extract_features(train, model, batch_size)
    X_test, y_test = extract_features(test, model, batch_size)

    #####################
    ## Votre code ici  ##
    #####################
    # Apprentissage et évaluation des SVM à faire
    svm = LinearSVC(C=1.0)
    print('Apprentissage des SVM')
    svm.fit(X_train, y_train)
    accuracy = svm.score(X_test, y_test)
    ####################n
    ##      FIN        #
    ####################
    print('Accuracy = %f' % accuracy)


In [ ]:
main("15SceneData", 8)